In [9]:
# Import packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

# Data Cleaning
## Cleaning file: "PrimeTransactions"

In [10]:
import csv

def row_count(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        
        # Skip the header
        next(reader, None)
        
        count = sum(1 for row in reader)
    return count

true_count = row_count('PrimeTransactions.csv')
print(f"Record count: {true_count:,}")

Record count: 11,027


In [11]:
#Eliminated the print statement to save space in the notebook
headers = pd.read_csv('PrimeTransactions.csv', nrows=0).columns.tolist()

<details>
<summary><b>Click to expand: Analysis of Chosen Columns</b></summary>
    
### Classifying useful columns
#### Identifiers & Financials

1. award_id_piid: Keep. The "Bridge" key to link with Subawards.
2. modification_number: Keep. Crucial for measuring scope creep.
3. federal_action_obligation: Keep. Your primary Y-variable (spending event).
4. total_dollars_obligated: Keep. Cumulative spending to date.
5. current_total_value_of_award: Keep. What the gov is currently committed to pay.
6. base_and_all_options_value: Keep. The "Potential" total value.

#### Project Dates & Performance

1. action_date: Keep. When this specific financial transaction happened.
2. period_of_performance_start_date: Keep. The "Project Birth" date.
3. period_of_performance_current_end_date: Keep. The active deadline.
4. period_of_performance_potential_end_date: Keep. Used for "Runway" calculation.
5. solicitation_date: Keep. Crucial for "Procurement Lead Time."

#### Agency & Office Details

1. awarding_agency_name: Keep. Confirms Department of Energy.
2. awarding_sub_agency_name: Keep. Best for grouping (e.g., Office of Science).

#### Accounting Metadata

1. recipient_name: Keep. Primary identification of the performer.

#### Place of Performance

1. primary_place_of_performance_city_name: Keep. Good for city-level spending.
2. primary_place_of_performance_state_name: Keep. Essential for Tableau mapping.

#### Contract Type & Classification

1. type_of_contract_pricing: Keep. Fixed Price vs. Cost Plus (Risk Factor).
2. transaction_description: Keep. Best technical detail in the prime file.
3. action_type: Keep. New Award vs. Extension vs. Funding Change.
4. product_or_service_code_description: Keep. Broad grouping (Physical vs. Service).
5. naics_description: Keep. Detailed industry classification.

#### Competition & Regulatory

1. extent_competed: Keep. Core variable for competition analysis.
2. type_of_set_aside: Keep. Tells you if work was reserved for small/disadvantaged businesses.
3. number_of_offers_received: Keep. Precise metric for competition intensity.

#### Business Ownership (Control Variables)

1. contracting_officers_determination_of_business_size: Keep. Best way to identify "Small Business" status.
   
</details>

In [12]:
prime_cols = [
    'award_id_piid', 
    'modification_number',
    'federal_action_obligation',
    'total_dollars_obligated',
    'current_total_value_of_award',
    'base_and_all_options_value',
    'action_date',
    'period_of_performance_start_date',
    'period_of_performance_current_end_date',
    'period_of_performance_potential_end_date',
    'solicitation_date',
    'awarding_agency_name',
    'awarding_sub_agency_name',
    'recipient_name',
    'primary_place_of_performance_city_name',
    'primary_place_of_performance_state_name',
    'type_of_contract_pricing',
    'transaction_description',
    'action_type',
    'product_or_service_code_description',
    'naics_description',
    'extent_competed',
    'type_of_set_aside',
    'number_of_offers_received',
    'contracting_officers_determination_of_business_size'
]

prime_clean = pd.read_csv('PrimeTransactions.csv', usecols=prime_cols)

print(f"Dataset Dimensions: {prime_clean.shape}")
prime_clean.head()

Dataset Dimensions: (11027, 25)


,award_id_piid,modification_number,federal_action_obligation,total_dollars_obligated,current_total_value_of_award,base_and_all_options_value,action_date,period_of_performance_start_date,period_of_performance_current_end_date,period_of_performance_potential_end_date,...,primary_place_of_performance_state_name,type_of_contract_pricing,transaction_description,action_type,product_or_service_code_description,naics_description,extent_competed,type_of_set_aside,number_of_offers_received,contracting_officers_determination_of_business_size
0,89243323PFE000652,P00001,0.00,1.600000e+04,1.600000e+04,0.00,2024-01-02,2023-06-22,2023-07-22,2023-07-22 00:00:00,...,WEST VIRGINIA,FIRM FIXED PRICE,CLOSEOUT VASP 6 LICENSE PERPETUAL SOFTWARE LIC...,CLOSE OUT,IT AND TELECOM - BUSINESS APPLICATION SOFTWARE...,OTHER COMPUTER RELATED SERVICES,NOT COMPETED,NO SET ASIDE USED.,1.0,OTHER THAN SMALL BUSINESS
1,89243324PFE000700,0,89518.00,8.951800e+04,8.951800e+04,89518.00,2023-12-15,2023-12-15,2024-12-31,2024-12-31 00:00:00,...,OREGON,FIRM FIXED PRICE,"NETL ALBANY, OR BUILDING 28 ARCHITECTURAL & HV...",NaN,ARCHITECT AND ENGINEERING- GENERAL: OTHER,ENGINEERING SERVICES,NOT COMPETED,NO SET ASIDE USED.,1.0,SMALL BUSINESS
2,89303024PAU000047,0,31750.00,3.175000e+04,3.175000e+04,31750.00,2024-08-29,2024-08-29,2025-02-28,2025-02-28 00:00:00,...,DISTRICT OF COLUMBIA,FIRM FIXED PRICE,THIS IS A FIRM-FIXED PRICE PURCHASE ORDER WITH...,NaN,SAFETY AND RESCUE EQUIPMENT,OTHER PROFESSIONAL EQUIPMENT AND SUPPLIES MERC...,NOT COMPETED UNDER SAP,NO SET ASIDE USED.,1.0,SMALL BUSINESS
3,DEAC0206CH11357,1001,16431217.58,1.504272e+10,1.558112e+10,18677709.67,2024-08-28,2006-07-31,2026-09-30,2026-09-30 00:00:00,...,ILLINOIS,COST PLUS AWARD FEE,PERFORMANCE-BASED CONTRACT FOR MANAGEMENT AND ...,FUNDING ONLY ACTION,OPER OF GOVT R&D GOCO FACILITIES,"RESEARCH AND DEVELOPMENT IN THE PHYSICAL, ENGI...",FULL AND OPEN COMPETITION,NO SET ASIDE USED.,1.0,OTHER THAN SMALL BUSINESS
4,89233121PNA000132,P00005,60000.00,1.186836e+05,1.186836e+05,34800.00,2024-02-14,2021-09-20,2024-09-19,2026-09-19 00:00:00,...,VIRGINIA,FIRM FIXED PRICE,PARTS AND REPAIR OF L3-MANUFACTURED BNVD-1531,FUNDING ONLY ACTION,"NIGHT VISION EQUIPMENT, EMITTED AND REFLECTED ...","SEARCH, DETECTION, NAVIGATION, GUIDANCE, AERON...",COMPETED UNDER SAP,NO SET ASIDE USED.,3.0,SMALL BUSINESS


## Cleaning file "Subawards"

In [13]:
def row_count(filename):
    with open(filename, 'r', encoding='utf-8') as f:
        reader = csv.reader(f)
        
        # Skip the header
        next(reader, None)
        
        count = sum(1 for row in reader)
    return count

true_count = row_count('Subawards.csv')
print(f"Record count: {true_count:,}")

Record count: 9,043


In [14]:
#Eliminated the print statement to save space in the notebook
headers = pd.read_csv('Subawards.csv', nrows=0).columns.tolist()
#headers

<details>
<summary><b>Click to expand: Analysis of Chosen Columns</b></summary>
    
### Classifying useful columns
#### Link Keys
1. prime_award_piid: To match the award_id_piid in the Prime_Transactions file.
2. prime_award_unique_key: Secondary backup key for merging.

#### Subaward Financials
1. subaward_amount: The amount given to the sub-contractor.
2. prime_award_amount: Essential to calculate what percentage of the total project is being outsourced.
3. subaward_action_date: When the sub-contract was issued.

##### Subaward Operations & Context

1. prime_award_base_action_date_fiscal_year:
2. prime_award_latest_action_date: Useful to see the "age" of the prime contract at the moment the subaward was issued. **PENDING**
3. prime_award_period_of_performance_start_date: Test to know the "arrival" and "departure" of a subcontractor
4. prime_award_period_of_performance_current_end_date: Defines the current funded "deadline" for the work.
5. prime_award_period_of_performance_potential_end_date: Defines the total "Runway" or planned lifecycle of the engineering project.

#### Prime Agency Info

1. prime_award_awarding_agency_name: Keep. Confirms this is Department of Energy.
2. prime_award_awarding_sub_agency_name: Keep. Essential for grouping (e.g., Office of Science vs. NNSA).

#### Awardee Info

1. prime_awardee_name: Keep. Identifies the main contractor.
2. prime_awardee_city_name: Keep. Useful for city-level spending trends.
3. prime_awardee_state_name: Keep. Crucial for Tableau mapping.
4. prime_awardee_business_types: Keep. Tells you the "Risk Profile" of the prime (e.g., Non-profit, Small Business).

#### Project details

1. prime_award_project_title: Keep. Provides the "Human-readable" name of the project.
2. prime_award_naics_description: Keep. Identifies the industry of the Prime.

#### Subaward Details

1. subaward_number: Keep. To track specific tasks within a PIID.
2. subaward_amount: Keep. The "Money" variable for your sub-analysis.
3. subaward_action_date: Keep. The "Arrival" date of the specialist.
4. subawardee_name: Keep. Identifies who is actually doing the outsourced work.
5. subawardee_state_name: Keep. Shows the geographical "reach" of the subcontracting.
6. subawardee_business_types: Keep. Vital for analyzing if subcontracts are going to small businesses.
7. subaward_description: Keep. Often provides the most technical engineering detail.

</details>

In [18]:
subawards_cols = [
    'prime_award_piid', 
    'prime_award_unique_key',
    'subaward_amount',
    'prime_award_amount',
    'subaward_action_date',
    'prime_award_base_action_date_fiscal_year',
    'prime_award_latest_action_date',
    'prime_award_period_of_performance_start_date',
    'prime_award_period_of_performance_current_end_date',
    'prime_award_period_of_performance_potential_end_date',
    'prime_award_awarding_agency_name',
    'prime_award_awarding_sub_agency_name',
    'prime_awardee_name',
    'prime_awardee_city_name',
    'prime_awardee_state_name',
    'prime_awardee_business_types',
    'prime_award_project_title',
    'prime_award_naics_description',
    'subaward_number',
    'subaward_amount',
    'subaward_action_date',
    'subawardee_name',
    'subawardee_state_name',
    'subawardee_business_types',
    'subaward_description'
]

subawards_df = pd.read_csv('Subawards.csv', usecols=subawards_cols)

print(f"Dataset Dimensions: {subawards_df.shape}")
subawards_df.head()

Dataset Dimensions: (9043, 23)


,prime_award_unique_key,prime_award_piid,prime_award_amount,prime_award_base_action_date_fiscal_year,prime_award_latest_action_date,prime_award_period_of_performance_start_date,prime_award_period_of_performance_current_end_date,prime_award_period_of_performance_potential_end_date,prime_award_awarding_agency_name,prime_award_awarding_sub_agency_name,...,prime_awardee_business_types,prime_award_project_title,prime_award_naics_description,subaward_number,subaward_amount,subaward_action_date,subawardee_name,subawardee_state_name,subawardee_business_types,subaward_description
0,CONT_AWD_DENA0001942_8900_-NONE-_-NONE-,DENA0001942,2.971210e+10,2013,2026-03-31,2013-01-08,2027-09-30,2027-09-30,Department of Energy (DOE),"ENERGY, DEPARTMENT OF",...,"For Profit Organization,Limited Liability Company",NaN,FACILITIES SUPPORT SERVICES,4300186319,109621.08,2024-03-27,"LATA-ATKINS TECHNICAL SERVICES, LLC",NEW MEXICO,"FOR-PROFIT ORGANIZATION,LIMITED LIABILITY COMPANY",SERVICES
1,CONT_AWD_DENA0001942_8900_-NONE-_-NONE-,DENA0001942,2.971210e+10,2013,2026-03-31,2013-01-08,2027-09-30,2027-09-30,Department of Energy (DOE),"ENERGY, DEPARTMENT OF",...,"For Profit Organization,Limited Liability Company",NaN,FACILITIES SUPPORT SERVICES,4300183202,37300.00,2023-10-01,"BOSTON GOVERNMENT SERVICES, LLC",TENNESSEE,"FOR-PROFIT ORGANIZATION,LIMITED LIABILITY COMPANY",SERVICES
2,CONT_AWD_DENA0001942_8900_-NONE-_-NONE-,DENA0001942,2.971210e+10,2013,2026-03-31,2013-01-08,2027-09-30,2027-09-30,Department of Energy (DOE),"ENERGY, DEPARTMENT OF",...,"For Profit Organization,Limited Liability Company",NaN,FACILITIES SUPPORT SERVICES,4300184430,83008.40,2023-12-06,"WILDFLOWER INTERNATIONAL, LTD.",NEW MEXICO,"MINORITY-OWNED BUSINESS,FOR-PROFIT ORGANIZATIO...",MATERIALS
3,CONT_AWD_DENA0001942_8900_-NONE-_-NONE-,DENA0001942,2.971210e+10,2013,2026-03-31,2013-01-08,2027-09-30,2027-09-30,Department of Energy (DOE),"ENERGY, DEPARTMENT OF",...,"For Profit Organization,Limited Liability Company",NaN,FACILITIES SUPPORT SERVICES,0000078836,42150.50,2024-04-10,DAVID EGERTON,TEXAS,"FOR-PROFIT ORGANIZATION,VETERAN OWNED BUSINESS",SERVICES
4,CONT_AWD_DENA0001942_8900_-NONE-_-NONE-,DENA0001942,2.971210e+10,2013,2026-03-31,2013-01-08,2027-09-30,2027-09-30,Department of Energy (DOE),"ENERGY, DEPARTMENT OF",...,"For Profit Organization,Limited Liability Company",NaN,FACILITIES SUPPORT SERVICES,4300183023,220889.60,2023-10-01,"BOSTON GOVERNMENT SERVICES, LLC",TENNESSEE,"FOR-PROFIT ORGANIZATION,LIMITED LIABILITY COMPANY",SERVICES


In [20]:
# Strip whitespace and force uppercase for both keys
prime_clean['award_id_piid'] = prime_clean['award_id_piid'].str.strip().str.upper()
subawards_df['prime_award_piid'] = subawards_df['prime_award_piid'].str.strip().str.upper()

In [22]:
# Find unique PIIDs in each
prime_piids = set(prime_clean['award_id_piid'])
sub_piids = set(subawards_df['prime_award_piid'])

# Calculate overlap
matches = sub_piids.intersection(prime_piids)
missing = sub_piids - prime_piids

print(f"Successful Connections: {len(matches)}")
print(f"Subawards with no matching Prime: {len(missing)}")

Successful Connections: 114
Subawards with no matching Prime: 12


#### Discarding Orphaned Data
Since we cant use the subawards with no matching Prime to calculate their Latency or Outsourcing Ratio, we are going to discard them for the analysis.

In [23]:
# Use the .isin() method to keep only matching records
clean_sub_df = subawards_df[subawards_df['prime_award_piid'].isin(prime_piids)].copy()

# Verify the drop
dropped_count = len(subawards_df) - len(clean_sub_df)
print(f"Dropped {dropped_count} orphaned subawards.")
print(f"New Subaward total: {len(clean_sub_df)}")

Dropped 2698 orphaned subawards.
New Subaward total: 6345


In [25]:
# Now run your logic check on the CLEANED dataframe
clean_sub_df['subaward_amount'] = pd.to_numeric(clean_sub_df['subaward_amount'], errors='coerce')
clean_sub_df['prime_award_amount'] = pd.to_numeric(clean_sub_df['prime_award_amount'], errors='coerce')

violations = clean_sub_df[clean_sub_df['subaward_amount'] > clean_sub_df['prime_award_amount']]
print(f"Number of violations found: {len(violations)}")

Number of violations found: 3


In [27]:
violations[['subawardee_name', 'subaward_amount', 'prime_award_amount', 'subaward_description']]

,subawardee_name,subaward_amount,prime_award_amount,subaward_description
6549,"TXA POWERSPORTS, INC.",151512.17,92613.02,"ALPHA LOGISTICS GROUP, INC."
6914,REDHORSE CORPORATION,151259.92,149664.00,BUSINESS SUPPORT SERVICES FOR THE VTO
9024,"GROUP14 ENGINEERING, PBC",67424.00,0.00,CX SERVICES


In [31]:
# Convert Prime Dates
prime_date_cols = ['action_date', 'period_of_performance_start_date', 
                   'period_of_performance_current_end_date']

for col in prime_date_cols:
    prime_clean[col] = pd.to_datetime(prime_clean[col], errors='coerce')

# Convert Subaward Dates
subawards_df['subaward_action_date'] = pd.to_datetime(subawards_df['subaward_action_date'], errors='coerce')

print("Dates converted to datetime objects.")

Dates converted to datetime objects.


In [33]:
# Fill missing text info with a placeholder
subawards_df['subaward_description'] = subawards_df['subaward_description'].fillna('No Description Provided')
prime_clean['type_of_set_aside'] = prime_clean['type_of_set_aside'].fillna('None/Unspecified')

In [34]:
prime_clean['recipient_name'] = prime_clean['recipient_name'].str.title()
subawards_df['subawardee_name'] = subawards_df['subawardee_name'].str.title()

In [36]:
# Create the final version by filtering FOR logical rows
# This automatically 'drops' the 3 violations we found
final_sub_df = clean_sub_df[clean_sub_df['subaward_amount'] <= clean_sub_df['prime_award_amount']].copy()

print(f"Final record count for analysis: {len(final_sub_df):,}")

Final record count for analysis: 6,342
